# LA-Breast preprocessing

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
assert (REPO_ROOT / "pyproject.toml").exists(), "Could not locate the repository root."
os.chdir(REPO_ROOT)  
sys.path.insert(0, str(REPO_ROOT))

## Load raw data

In [ ]:
from preprocessing.la_breast import LABreast, csv_save_path
from utils.preprocessing import birads_assessment, birads_mapping, breast_density, get_value

ds = LABreast()
ds.set_dataset_name("la-breast")
ds.clinical_df.shape, ds.clinical_df.head()

### Step 1 — map raw BI-RADS codes to harmonized labels

In [ ]:
ds.clinical_df["birads"] = ds.clinical_df["birads"].apply(lambda x: get_value(x, birads_assessment))
ds.clinical_df["birads"].value_counts(dropna=False)

### Step 2 — fold known-biopsy-proven (6) into highly-suggestive (5)

In [ ]:
ds.clinical_df["birads"] = ds.clinical_df["birads"].replace(birads_mapping[6], birads_mapping[5])
ds.clinical_df["birads"].value_counts(dropna=False)

### Step 3/4 — map breast density (ACR) and rename the column

In [ ]:
ds.clinical_df["acr"] = ds.clinical_df["acr"].apply(lambda x: get_value(int(x), breast_density) if pd.notna(x) else None)
ds.clinical_df.rename(columns={"acr": "breast density"}, inplace=True)
ds.clinical_df["breast density"].value_counts(dropna=False)

### Step 5 — keep only the columns needed to locate images and build findings

In [ ]:
keep_cols = [
    "patient", "roi", "birads", "breast density",
    "distancia x d0", "distancia y d0", "centro x d0", "centro y d0", "centro z d0", "d0",
    "d1", "d2", "d3", "d4", "d5",
    "centro x d1", "centro y d1", "centro z d1", "distancia x d1", "distancia y d1",
    "centro x d2", "centro y d2", "centro z d2", "distancia x d2", "distancia y d2",
    "centro x d3", "centro y d3", "centro z d3", "distancia x d3", "distancia y d3",
    "centro x d4", "centro y d4", "centro z d4", "distancia x d4", "distancia y d4",
    "centro x d5", "centro y d5", "centro z d5", "distancia x d5", "distancia y d5",
]
ds.clinical_df = ds.clinical_df[keep_cols]
ds.clinical_df.shape

### Step 6 — keep one random row per (patient, BI-RADS, breast density)

In [ ]:
dedup_df = ds._sample_one_row_per_clinical_group(ds.clinical_df, random_state=42)
dedup_df.shape

## Build exam records for one patient

In [ ]:
patient_id, patient_df = next(iter(dedup_df.groupby("patient")))
exams = ds.process_patient(patient_df)
exams

## Final processed CSV

In [ ]:
import json

final_df = pd.read_csv(csv_save_path)
print(f"deduplicated clinical rows: {len(dedup_df)} | rows in final csv: {len(final_df)}")
final_df.head()

In [ ]:
sample = final_df.iloc[0]
print(json.dumps(json.loads(sample['context']), indent=2))
print(json.dumps(json.loads(sample['findings']), indent=2))